In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

# =========================
# DRIVERS
# =========================
@dlt.table(name="drivers", comment="Dados de pilotos limpos e enriquecidos")
@dlt.expect_all({
    "valid_driver_id": "driver_id IS NOT NULL",
    "valid_name": "full_name IS NOT NULL"
})
def drivers():
    return (
        spark.read.table("f1_lakehouse.bronze.drivers_raw")
        .select(
            col("driverId").alias("driver_id"),
            col("givenName").alias("forename"),
            col("familyName").alias("surname"),
            concat(col("givenName"), lit(" "), col("familyName")).alias("full_name"),
            col("nationality"),
            to_date(col("dateOfBirth"), "yyyy-MM-dd").alias("date_of_birth"),
            col("url"),
            col("_ingested_at"),
            col("_source"),
            col("_season").alias("season")
        )
        .filter(col("driver_id").isNotNull())
        .dropDuplicates(["driver_id", "season"])
    )


# =========================
# CONSTRUCTORS
# =========================
@dlt.table(name="constructors", comment="Dados de construtores limpos")
@dlt.expect_all({
    "valid_constructor_id": "constructor_id IS NOT NULL",
    "valid_name": "name IS NOT NULL"
})
def constructors():
    return (
        spark.read.table("f1_lakehouse.bronze.constructors_raw")
        .select(
            col("constructorId").alias("constructor_id"),
            col("name"),
            col("nationality"),
            col("_ingested_at"),
            col("_source"),
            col("_season").alias("season")
        )
        .filter(col("constructor_id").isNotNull())
        .dropDuplicates(["constructor_id", "season"])
    )


# =========================
# RACES
# =========================
@dlt.table(name="races", comment="Dados de corridas limpos", partition_cols=["season"])
@dlt.expect_all({
    "valid_round": "round IS NOT NULL",
    "valid_date": "race_date IS NOT NULL"
})
def races():
    return (
        spark.read.table("f1_lakehouse.bronze.races_raw")
        .select(
            col("_season").alias("season"),
            col("round").cast("int"),
            col("circuitId").alias("circuit_id"),
            col("raceName").alias("race_name"),
            to_date(col("date"), "yyyy-MM-dd").alias("race_date"),
            col("time").alias("race_time"),
            col("url"),
            col("_ingested_at"),
            col("_source")
        )
        .filter(col("round").isNotNull())
        .dropDuplicates(["season", "round"])
    )


# =========================
# RESULTS
# =========================
@dlt.table(name="results", comment="Resultados enriquecidos", partition_cols=["season"])
@dlt.expect_all({
    "valid_driver_id": "driver_id IS NOT NULL",
    "valid_points": "points >= 0",
    "valid_position": "final_position IS NULL OR final_position > 0"
})
def results():
    results_df = spark.read.table("f1_lakehouse.bronze.results_raw").alias("res")
    drivers_df = dlt.read("drivers").alias("drv")
    races_df   = dlt.read("races").alias("rc")

    return (
        results_df
        .join(drivers_df, col("res.driverId") == col("drv.driver_id"), "left")
        .join(
            races_df,
            (col("res._season") == col("rc.season")) & (col("res.raceId") == col("rc.round")),
            "left"
        )
        .select(
            col("res.raceId").alias("race_round"),
            col("res.driverId").alias("driver_id"),
            col("res.constructorId").alias("constructor_id"),
            col("res.grid").cast("int").alias("grid_position"),
            col("res.position").cast("int").alias("final_position"),
            col("res.positionText").alias("position_text"),
            col("res.points").cast("float"),
            col("res.laps").cast("int"),
            col("res.time").alias("finish_time"),
            col("res.milliseconds").cast("long"),
            col("res.fastestLap").cast("int").alias("fastest_lap"),
            col("res.fastestLapTime").alias("fastest_lap_time"),
            col("res.fastestLapSpeed").cast("float").alias("fastest_lap_speed"),
            col("res.status").alias("status"),                  # ✅ era statusId
            col("res._season").alias("season"),
            col("drv.full_name").alias("driver_name"),
            col("rc.race_name"),
            col("rc.race_date"),
            col("res._ingested_at").alias("_ingested_at"),
            col("res._source").alias("_source")
        )
        .filter(col("driver_id").isNotNull())
    )